### File used to merge outputs from NLP models on structured data.

In [ ]:
import pandas as pd

1. Check if toxic scores are included in political_interactions

In [ ]:
flat_toxic = pd.read_csv('../clean_data/flat_toxic_interactions.csv', index_col=0)
flat_political = pd.read_csv('../clean_data/flat_political_interactions.csv')

In [ ]:
flat_toxic = flat_toxic.sort_values("toxicity_level", ascending=False).drop_duplicates(subset="text", keep="first")
test = pd.merge(flat_political, flat_toxic, on='text', how='left', indicator=True)
print(test['_merge'].value_counts())
# Tous les commentaires de toxic sont inclus dans les commentaires politiques. 
test = test.drop(columns=['_merge'])

2. Merge Left-Right in political interactions.

In [ ]:
flat_left_right = pd.read_csv('../clean_data/flat_left_right_interactions.csv')
test = pd.merge(test, flat_left_right, on='text', how='left', indicator=True)
print(test['_merge'].value_counts())
# Tous les commentaires de left-right sont inclus dans les commentaires politiques. 
test = test.drop(columns=['_merge'])

3. Merge political interactions in all interactions 

In [ ]:
flat_interactions = pd.read_csv('../clean_data/flat_all_interactions.csv')
test = pd.merge(flat_interactions, test, on='text', how='left', indicator='political')

In [ ]:
print(test['political'].value_counts())
# All political interactoins are in flat interactions
test['political'] = test['political'].replace({
    'left_only': 0,
    'both':1
    }).cat.remove_unused_categories()
print(test['political'].value_counts())

4. Merge all flat interactions on all structured interactions

In [ ]:
struct_interactions = pd.read_csv('../clean_data/struc_all_interactions.csv')
struct_interactions = struct_interactions.rename(columns={
    'text':'user_text',
    'text_inter':'inter_text'
})
struct_interactions = struct_interactions.drop(columns=['pfp_inter', 'after', 'date_ins'])

In [ ]:
struct_panel = pd.merge(struct_interactions, test, left_on='user_text', right_on='text', how='left')
struct_panel = struct_panel.rename(columns={
    'toxicity_level':'user_toxicity_level', 
    'left_right':'user_left_right', 
    'political':'user_political'
    })
struct_panel = struct_panel.drop(columns=['text'])

In [ ]:
struct_panel = pd.merge(struct_panel, test, left_on='inter_text', right_on='text', how='left')
struct_panel = struct_panel.rename(columns={
    'toxicity_level':'inter_toxicity_level', 
    'left_right':'inter_left_right', 
    'political':'inter_political'
    })
struct_panel = struct_panel.drop(columns=['text'])

In [ ]:
struct_panel = pd.merge(struct_panel, test, left_on='post_text', right_on='text', how='left')
struct_panel = struct_panel.rename(columns={
    'toxicity_level':'post_toxicity_level', 
    'left_right':'post_left_right', 
    'political':'post_political'
    })
struct_panel = struct_panel.drop(columns=['text'])

In [ ]:
struct_panel.columns

In [ ]:
struct_panel.to_csv('../clean_data/struct_panel.csv', index=False)